# Feature Engineering
### Predict perfromance of real 3E cells using machine learning. This notebook imports data and proceses it to be ready for running it through different models

In [2]:
import pandas as pd
pd. __version__

'2.0.2'

In [1]:
#import libraries

import numpy as np
import matplotlib.pyplot as plt
import matplotlib
%matplotlib qt
import itertools
import os
import datetime

from scipy import integrate
import scipy.stats
import pickle
import mgzip
import pandas as pd
import importlib

from scipy.interpolate import interp1d
from scipy.ndimage import gaussian_filter1d
import math

# from statsmodels.graphics.tsaplots import plot_acf

import time


import subprocess
from subprocess import call
from xlsx2csv import Xlsx2csv


In [5]:
#run just once to convert xlsx to csv

# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrCanrd-LFPCanrd-Li"
rootdir = r"..\ArbinResultData\Testing Data\CC-CV"
# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrMTI-LFPxr-LFPRE-TS-3\0.33C"
# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrMTI-LFPxr-Li-3.55V-032323-040623"
# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrCanrd-LFPCanrd-Li-032323-040623"
# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrMTI-LFPxr-Li-3.55V-Stepped"

for subdir, dirs, files in os.walk(rootdir):
    real_data = []

    strpath = subdir
    idx = strpath.rfind("channel_")
    if(idx == -1):
        continue
    
    text = strpath[idx+7:idx+10]
    sheet_num  = ""
    for i in text:
        if i.isdigit():
            sheet_num += i

    print(subdir)
    temp2 = strpath.find("0.")
    temp3 = strpath.find('C\\')
    c_rate = strpath[temp2:temp3]

    appended = False

    if 'IR' in strpath or "副本" in strpath:
        continue
    
    print(c_rate)
    for file in files:
        if "GITT" in os.path.join(subdir, file):
            continue
        elif "11.81ohm" in os.path.join(subdir, file):
            continue
        
        strpath = os.path.join(subdir, file)
        # if "channel_7_" in strpath:
            # continue
        if "csv" in file:
            continue
         
        print(strpath)
        # print(cell_num)


        csv = subdir + '/' + file[:-5] + ".csv"
        print(csv)

        temp = os.path.getmtime(strpath)
        dt_m = datetime.datetime.fromtimestamp(temp)
        compare_date =  datetime.datetime(2023, 3, 10)

        if(dt_m < compare_date):
            continue

        print(dt_m)
        # break
        # print(file)
        Xlsx2csv(strpath, outputencoding="utf-8").convert(csv, sheetname = 'Channel-' + sheet_num + '_1' )
        # call(['cscript.exe', 'ExcelToCsv.vbs', strpath, csv, ])
        
        # df[sheet_num]=pd.read_csv(csv)
    # break

del temp2, temp3, subdir, strpath

..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_2023_03_26_171036
0.94
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_2023_03_26_171036\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_Channel_15_Wb_1.xlsx
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_2023_03_26_171036/Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_Channel_15_Wb_1.csv
2023-04-18 13:51:23
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_2023_03_26_171036\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_Channel_15_Wb_2.xlsx
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_2023_03_26_171036/Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_Channel_15_Wb_2.csv
2023-04-18 13:52:10
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_032723_channel_15_2023_03_2

In [2]:
#LOAD IN REAL DATA

#directory to load data from
# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrCanrd-LFPCanrd-Li"
# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrMTI-LFPxr-Li-3.55V-032323-040623"
# rootdir = r"D:\akit2\Documents\Box Sync\Li\ArbinResultData\Testing Data\GrCanrd-LFPCanrd-Li-032323-040623"
rootdir = r"..\ArbinResultData\Testing Data\CC-CV"
cell_num = 1

#structure to store data
masterInput = []

for subdir, dirs, files in os.walk(rootdir):
    real_data = []

    strpath = subdir
    idx = strpath.rfind("channel_")
    if(idx == -1):
        continue
    
    text = strpath[idx+7:idx+10]
    sheet_num  = ""
    for i in text:
        if i.isdigit():
            sheet_num += i

    # print(subdir)
    temp2 = strpath.rfind(rootdir)
    temp3 = strpath.find('C\Jiayi')
    c_rate = strpath[temp2+len(rootdir)+1:temp3]
    if(len(c_rate) > 5):
        c_rate = "stepped"
    
    appended = False
    temp = 1
    print(cell_num, c_rate)
    for file in files:
        if "GITT" in os.path.join(subdir, file):
            continue
        elif "11.81ohm" in os.path.join(subdir, file):
            continue
        elif "SEM" in os.path.join(subdir, file):
            continue
       
        elif "021623_channel_17_2023_02_16_163747" in os.path.join(subdir, file):
            continue

        if "xlsx" in file:
            continue
        


        strpath = os.path.join(subdir, file)
        print(strpath)
        


        real_data = pd.read_csv(strpath)
        # real_data = pd.read_csv()
        real_data['cell_num'] = cell_num
        real_data['C-rate'] = c_rate

        real_data['Voltage'] = real_data['Voltage (V)'].copy()
        real_data['Current'] = real_data['Current (A)'].copy()
        real_data['Time'] = real_data['Step Time (s)'].copy()
        real_data['TotTime'] = real_data['Test Time (s)'].copy()
        real_data['NEP'] =  real_data['Aux_Voltage_1 (V)'].copy()
        real_data['Step_Index'] = real_data['Step Index'].copy()
        real_data['Cycle_Index'] = real_data['Cycle Index'].copy()
        real_data['Charge_Capacity'] = real_data['Charge Capacity (Ah)']
        real_data['Discharge_Capacity'] = real_data['Discharge Capacity (Ah)']
        real_data['Charge_Energy'] = real_data['Charge Energy (Wh)']

        real_data = real_data[['Voltage', 'Current', 'Time', 'NEP', 'TotTime', 'Step_Index', 'Cycle_Index', 'cell_num', 'C-rate', 'Charge_Capacity', 'Discharge_Capacity', 'Charge_Energy']]

        if "Activation cycles" in os.path.join(subdir, file):
            print(type(masterInput[0]))
            real_data['cell_num'] = cell_num - 1
            masterInput[cell_num-2] = pd.concat((real_data, masterInput[cell_num-2] ), axis = 0)
            appended = False
        else:
            masterInput.append(real_data)
            appended = True

    if(appended):   
        cell_num+=1

print("complete")

#run once

with mgzip.open("Real Data\data021824", 'wb') as f:
    pickle.dump(pd.concat(masterInput), f)


1 0.94
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_051923_channel_22_2023_05_19_180402\Jiayi_GraphiteMTI_LFPXR_C100_94C_051923_channel_22_Channel_22_Wb_1.csv
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_051923_channel_22_2023_05_19_180402\Jiayi_GraphiteMTI_LFPXR_C100_94C_051923_channel_22_Channel_22_Wb_2.csv
..\ArbinResultData\Testing Data\CC-CV\0.94C\Jiayi_GraphiteMTI_LFPXR_C100_94C_051923_channel_22_2023_05_19_180402\Jiayi_GraphiteMTI_LFPXR_C100_94C_051923_channel_22_Channel_22_Wb_3.csv
2 1
..\ArbinResultData\Testing Data\CC-CV\1C\Jiayi_GraphiteMTI_LFPXR_C101C_052323_channel_16_2023_05_23_152329\Jiayi_GraphiteMTI_LFPXR_C101C_052323_channel_16_Channel_16_Wb_1.csv
..\ArbinResultData\Testing Data\CC-CV\1C\Jiayi_GraphiteMTI_LFPXR_C101C_052323_channel_16_2023_05_23_152329\Jiayi_GraphiteMTI_LFPXR_C101C_052323_channel_16_Channel_16_Wb_2.csv
..\ArbinResultData\Testing Data\CC-CV\1C\Jiayi_GraphiteMTI_LFPXR_C101C_052323_channel_1

In [2]:
#load in data structure
# from doctest import master

#10/13 - double donut, low C-rate data
#10/27 - single donut, more c-rates
#11/18 - double donut, higher c-rates
#12/1 - double donut, lower c-rates
#1/12 - double donut, lower c-rates, correct n/p ratio, weird signals
#1/19 - 14-5 geometry for cathode, correct n/p ratio, ideally good signals
#1/19-poor - 12-5 geometry, noisy signals 
#1/25 - 14-5 geometry with more cycles
#2/6 - 14-5 geometry big dataset with more cycles and LFP RE
#2/15 - LFP RE
#2/15 - Li RE
#3/15-3.55 - 3.55V, activation cycle is modified
#3/15-LFP - same as 2/15 but with more data and cells
#3/15-Li - same as 2/15 but with more data and cells
#4/5-3.55 - new cells with 3.55V activation
#4/5-Li - Li RE with Canrd electrode
#4/19-Stepped - 3.55V, stepped data
#4/19-3.55 - 3.55V
#4/19-Canrd - Canrd Li, long cycles
#5/23-3.55, stepped

import datetime
date = datetime.date.today().strftime('%Y-%m-%d')
#date = "2023-06-22"

dataset_filename = "data110323"
# dataset_filename = "data041923-Canrd"
# dataset_filename = "data031523-3.55V"
# masterInput = pd.read_pickle(r"Real Data/"  + dataset_filename)

with mgzip.open(r"Real Data/"  + dataset_filename, 'rb') as f:
    masterInput = pickle.load(f)

#Enable these 6 lines after we get stepped data
with mgzip.open(r"Real Data\data110323-stepped", 'rb') as f:
    df = pickle.load(f)
maxCell = masterInput.cell_num.max()
df['cell_num'] = df['cell_num'] + maxCell
masterInput = pd.concat((masterInput, df), axis = 0)
del df

# masterBackup = masterInput.copy()
masterList = masterInput.groupby('cell_num')['C-rate'].first()
masterList

cell_num
1        0.33
2        0.33
3        0.33
4        0.33
5         0.5
6         0.5
7         0.5
8         0.5
9         0.5
10        0.8
11        0.8
12        0.8
13        0.8
14        0.8
15        0.8
16        0.8
17        0.8
18        0.8
19        0.8
20        0.8
21        0.8
22        0.8
23        0.8
24        0.8
25        0.8
26        0.8
27        0.8
28        0.8
29          1
30          1
31          1
32          1
33          1
34          1
35          1
36          1
37          1
38          1
39          1
40          1
41          1
42          1
43          1
44          1
45          1
46          1
47          1
48          1
49          1
50          1
51          1
52    stepped
53    stepped
54    stepped
55    stepped
56    stepped
57    stepped
Name: C-rate, dtype: object

We now start removing poor cells. For this, you will need to run through the subsequent blocks of code until reaching the xCC, yCC generation block. For cells that fail at any point (generating the voltage reference curve for overpotential calculation, or when calculating xCC). Note that if you remove cells, then run the blocks and encounter some failures, you will have to recalculate their original cell index values since they get reordered during the removal process.

In [3]:
#determine cells to remove

plt.close('all')

#cherrypicking, 3/15-LFP
# drop_list = [2,3,4,5,6,7,11,16,18,19,21]

#2/6 
# drop_list = [2, 3, 4, 6, 19]

#12/8 - LFP RE
# drop_list = [3,10,11,12]

#3/15 3.55V activation
# drop_list = [5,8,10,12]

#3/15 Li Canrd
# drop_list = [1]

# drop_list = []

#4/19 3.55 activation with stepped
# drop_list = [8, 9, 10,12]

#4/19 Li Canrd
# drop_list = [1,3]

#5/2 3.55 activation with stepped
# drop_list = [8, 9, 10,12, 16]

#5/23 3.55 activation with stepped

#6/18
#drop_list = [8,10,12,24,36,42,43]


#6/22
#drop_list = [8,10,12,16,23,24,36,42,43]

#Before we plot, we empty the drop list, and after seeing the plot, decide which one we need to dispose

drop_list = [45,50]

masterList = masterInput.groupby('cell_num')['C-rate'].first()
masterList

masterList.drop(drop_list , inplace = True)
masterInput = masterInput[~masterInput['cell_num'].isin(drop_list)]

i = 1
for k in drop_list:
    val = k - i
    masterInput.loc[pd.Index(masterInput['cell_num'] > val), 'cell_num'] =  masterInput.loc[:, 'cell_num'][masterInput['cell_num'] > val] - 1
    print(val)
    i+=1
# masterInput['cell_num'][masterInput['cell_num'] > 5] =  masterInput['cell_num'][masterInput['cell_num'] > 5] - 1

masterList = masterInput.groupby('cell_num')['C-rate'].first()
masterList

44
48


cell_num
1        0.33
2        0.33
3        0.33
4        0.33
5         0.5
6         0.5
7         0.5
8         0.5
9         0.5
10        0.8
11        0.8
12        0.8
13        0.8
14        0.8
15        0.8
16        0.8
17        0.8
18        0.8
19        0.8
20        0.8
21        0.8
22        0.8
23        0.8
24        0.8
25        0.8
26        0.8
27        0.8
28        0.8
29          1
30          1
31          1
32          1
33          1
34          1
35          1
36          1
37          1
38          1
39          1
40          1
41          1
42          1
43          1
44          1
45          1
46          1
47          1
48          1
49          1
50    stepped
51    stepped
52    stepped
53    stepped
54    stepped
55    stepped
Name: C-rate, dtype: object

In [4]:
#reorder dataframe and set cell_num as index

masterInput.set_index(['cell_num'], inplace = True)
masterInput.dropna(inplace = True, axis = 1)
masterInput.dropna(inplace = True, axis = 0)

masterInput.sort_index(inplace=True)
masterInput = masterInput.groupby(['cell_num']).apply(pd.DataFrame.sort_values, 'TotTime')
masterInput = masterInput.droplevel(1)
totalCells = max(masterInput.index)


In [5]:
#obtain activation cycle data


referenceRealData = pd.DataFrame(columns = ['Capacity', 'Energy', 'Voltage'] )

cell_params = {}
cell_params["nom_cap"] = 0.0031
cell_params["nom_energy"] = cell_params["nom_cap"]*2
fRealReference = {}
plt.figure()

idx = (masterInput['Step_Index'] == 5) & (masterInput['TotTime'] > 100000)
for i in range(1, totalCells+1,1):
    
    # idx = (masterInput['Step_Index'] >= 5) & (masterInput['Step_Index'] <= 9)
    # plt.plot(masterInput[idx]['TotTime'].loc[i], masterInput[idx]['Voltage'].loc[i])
    # break
    try:
    
        
        # print(np.where(masterInput.loc[i]['TotTime'].diff() > 100000))
        # plt.plot(np.arange(0, len(masterInput.loc[i]['TotTime'])), masterInput.loc[i]['TotTime']) 
        
        idx = (masterInput['Step_Index'] == 5) & (masterInput['TotTime'] > 100000)
        temp_data = masterInput[idx].loc[i]


        temp = np.where(temp_data['TotTime'].diff() > 20000)[0][0]
        temp_data = temp_data.iloc[temp:-1]


        referenceRealData = pd.DataFrame(columns = ['Capacity', 'Energy', 'Voltage'] )
        referenceRealData['Capacity'] = temp_data.Charge_Capacity.to_numpy()
        referenceRealData['Energy'] = temp_data.Charge_Energy.to_numpy()
        # referenceRealData['Throughput'] = 0
        referenceRealData['Voltage'] = temp_data['Voltage'].to_numpy()
        temp = referenceRealData.Capacity/cell_params["nom_cap"]*100

        fRealReference[i, 0] = interp1d(temp, referenceRealData.Voltage, kind = 'cubic', fill_value=(min(referenceRealData.Voltage), max(referenceRealData.Voltage)), bounds_error = False)
        vals, idx = np.unique(referenceRealData.Voltage, return_index=True)
        # plt.plot(vals, temp[idx])
        fRealReference[i, 1] = interp1d(vals, temp[idx], kind = 'cubic', fill_value='extrapolate')

        # plt.figure()
        plt.plot(temp, referenceRealData.Voltage, label = i)
        print(i)
    except Exception as e:
        print('failed ' + str(i))
        print(e)

plt.xlabel('SOC')
plt.ylabel('V')
plt.title('Activation Cycles')
plt.legend()

del temp, temp_data

1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55


In [6]:
#use this to check there are no missing cell indices
masterInput.index.get_level_values(0)

Index([ 1,  1,  1,  1,  1,  1,  1,  1,  1,  1,
       ...
       55, 55, 55, 55, 55, 55, 55, 55, 55, 55],
      dtype='int64', name='cell_num', length=77993354)

In [7]:
#create "cycle" index and fix time measurements per cycle


masterInput = masterInput[masterInput['Step_Index'] > 9].copy()
#10 is charge
#check after discharge rest - use for splitting cycles
rest_step = 14
# rest_step = 16

rest_dynamic = 17

masterInput['cycle'] = -1

# groups = masterInput.groupby('cell_num')
for cell_num in range(1, totalCells+1,1):

    group = masterInput.loc[cell_num].copy().reset_index()


    if(masterList[cell_num] == 'stepped' ):
        #be carefullll
        if cell_num == 12:
            use_step = rest_dynamic + 1
        else:
            use_step = rest_dynamic
    else:
        use_step = rest_step
    

    #split at transition from rest to charge
    split_indices = group[group.Step_Index == use_step].index

    cycle = 1
    startCycle = 0
    group['cycle'] = -1
    group['Time'] = 0

    #set correct cycles and readjust time
    for idx in split_indices:
    
        group.iloc[startCycle:idx+1, group.columns.get_loc("cycle")] = cycle
        group.iloc[startCycle:idx+1, group.columns.get_loc("Time")]  = group.TotTime[startCycle:idx+1] - min( group.TotTime[startCycle:idx+1] )

        startCycle = idx+1
        cycle +=1
    

    masterInput.loc[cell_num, 'cycle'] = group['cycle'].values
    masterInput.loc[cell_num,'Time'] = group['Time'].values

del group, split_indices
masterInput.set_index(['cycle'], append=True, inplace = True)

In [8]:
#eliminate unnecessary cycles
removeCycles = [-1,1]

#try removing cycles 100 and onward
removeCycles = removeCycles + [i for i in range(101, 5000, 1)]

masterInput = masterInput[~masterInput.index.get_level_values(1).isin(removeCycles)]


In [9]:
#obtain capacity and energy measurements

#in Ah
# temp = masterInput.groupby(['cell_num', 'cycle']).apply(lambda g: abs(integrate.cumulative_trapezoid(g.Current, g.Time, initial = 0))/3600)
# masterInput['Capacity'] = pd.concat(pd.DataFrame(i) for i in temp).to_numpy()
masterInput['Capacity'] = masterInput['Charge_Capacity'].copy()

#in Wh
# temp = masterInput.groupby(['cell_num', 'cycle']).apply(lambda g: abs(integrate.cumulative_trapezoid(g.Capacity, g.Voltage, initial = 0))/3600)
# masterInput['Energy'] = pd.concat(pd.DataFrame(i) for i in temp).to_numpy()
masterInput['Energy'] = masterInput['Charge_Energy'].copy()

#throughput is in units of Ah
throughput = masterInput.groupby('cell_num')['Capacity'].apply(lambda x: x.diff().abs().cumsum())
throughput.rename("Throughput", inplace= True)
throughput = throughput.droplevel(1)
    
masterInput = pd.concat([masterInput, throughput], axis = 1)

del throughput

In [10]:
#drop cycles that are less than 80% max capacity

temp = []
for cell_num in masterList.keys():
    removeCycles = []
    vals = masterInput.loc[cell_num].groupby(['cycle']).max()['Capacity']
    nom = vals.values[0]


    removeCycles = vals[vals/nom*100 < 80].index
    removeCycles = np.append(removeCycles, [-1])
    removeCycles = removeCycles.tolist()
    removeCycles
    
    temp.append(masterInput[(masterInput.index.get_level_values(0) == cell_num) & (~masterInput.index.get_level_values(1).isin(removeCycles))])

    if(np.any(vals<20)):
        print('Strange capacity', cell_num, masterInput.loc[cell_num].index.unique().max())


masterInput = pd.concat(temp)
del temp

Strange capacity 1 100
Strange capacity 2 100
Strange capacity 3 100
Strange capacity 4 100
Strange capacity 5 100
Strange capacity 6 88
Strange capacity 7 100
Strange capacity 8 100
Strange capacity 9 86
Strange capacity 10 100
Strange capacity 11 100
Strange capacity 12 100
Strange capacity 13 81
Strange capacity 14 98
Strange capacity 15 57
Strange capacity 16 70
Strange capacity 17 100
Strange capacity 18 16
Strange capacity 19 100
Strange capacity 20 100
Strange capacity 21 36
Strange capacity 22 61
Strange capacity 23 100
Strange capacity 24 100
Strange capacity 25 62
Strange capacity 26 62
Strange capacity 27 87
Strange capacity 28 100
Strange capacity 29 100
Strange capacity 30 100
Strange capacity 31 100
Strange capacity 32 82
Strange capacity 33 72
Strange capacity 34 81
Strange capacity 35 100
Strange capacity 36 66
Strange capacity 37 100
Strange capacity 38 57
Strange capacity 39 100
Strange capacity 40 28
Strange capacity 41 100
Strange capacity 42 30
Strange capacity 43 

In [11]:
#use this to check there are no missing cell indices
masterInput.index.get_level_values(0).unique()

Index([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
       37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
       55],
      dtype='int64', name='cell_num')

### After processing, generate xCC and yCC. The data is limited to anode potential < 0.1V. Irregular cycles are dropped, and the anode potential is smoothed. Usually, if a cell is not good, the cell index will also get dropped at this step. 

In [12]:
#generate CC features

# dataBackup = masterInput.copy()



masterInput = masterInput.loc[:,~masterInput.columns.duplicated()]

#eps: error for current
#rolling_n: size of window for rolling mean
#sigma: size of standard deviation for gaussian filter
eps = 1e-3
rolling_n = 25
sigma = 40

cell_num = 2
cycle = 3

masterInput =  masterInput[masterInput['Current'] > 0]
# masterInput = masterInput[masterInput['NEP'] > -0.05]

#watch out for this CV removal step, for each c-rate the mode is different

if("stepped" in masterList.keys()):
    print("contains stepped protocol cells, check masterInput filtering mechanism")

# masterInput = masterInput[masterInput.groupby(['cell_num'], group_keys=False).apply(lambda d: abs(d.Current - d.Current.mode().to_numpy()) < eps)]


#use for stepped cells
masterInput = masterInput[masterInput.groupby(['cell_num'], group_keys=False).apply(lambda d: abs(d.Voltage - 3.6) > eps)]

masterInput = masterInput[masterInput['Voltage'] > 3.3]

if((masterInput['NEP'] < -3.428).sum() > 1e6):
    # masterInput = masterInput[masterInput['NEP']+3.428 < 0.1]
    masterInput['NEP'] = masterInput['NEP']+3.428 
else:
    # masterInput = masterInput[masterInput['NEP']  < 0.1]
    print('using Li reference')
    masterInput = masterInput[masterInput['NEP'] > -0.15]

masterInput.replace([np.inf, -np.inf], np.nan, inplace=True)
masterInput.dropna(inplace = True, axis = 0)

#drop cycles with strange data
#should be fixed later
try:
    values = masterInput['TotTime'].groupby('cell_num').diff().fillna(1).values
    idx = np.where((values > 1.5) & (values  < 1000))
    if(len(idx) != 1):
        masterInput.drop(index= masterInput.iloc[idx].index.unique(), inplace = True)
except Exception as e:
    print(e)

    
#cycles that start less than 3.3 V are typically wrong
# values = masterInput.groupby(['cell_num', 'cycle'])['Voltage'].values

# idx = np.where(masterInput['Voltage'] < 3.32)[0]
# if(len(idx) != 1):
#     masterInput.drop(index= masterInput.iloc[idx].index.unique(), inplace = True)


# print(xCC.shape)
# print(yCC.shape)

# masterInput = masterInput.drop_duplicates(subset = ['Voltage', 'Capacity'])
masterInput = masterInput.groupby(['cell_num', 'cycle'], group_keys=False).apply(lambda d: d.drop_duplicates())

# 'Charge_Capacity', 'Discharge_Capacity'

#xCC = masterInput[['Voltage','Current','Throughput','Energy','Capacity','Time','TotTime']].copy()

xCC = masterInput[['Voltage','Current','Throughput','Energy','Capacity','Time']].copy()
yCC = masterInput['NEP'].copy()
try:
    masterInput.drop(['TotTime', 'Step_Index', 'Cycle_Index', 'C-rate','Charge_Capacity', 'Discharge_Capacity'], axis = 1, inplace=True)
except:
    print('already dropped masterInput columns')

    
#if(np.any(masterInput['NEP'] < -3.428)):
    #yCC = yCC + 3.428

# print(xCC.tail())

xCC.sort_index(inplace=True)
yCC.sort_index(inplace=True)

fig, axs = plt.subplots(1,2)
axs[0].plot(xCC.loc[cell_num,cycle]['Time'].to_numpy(), xCC.loc[cell_num,cycle]['Voltage'].to_numpy())
axs[1].plot(xCC.loc[cell_num,cycle]['Voltage'].to_numpy(),  yCC.loc[cell_num,cycle].to_numpy(), 'blue', label = 'original')


yCC = yCC.groupby(['cell_num', 'cycle'], sort = True).rolling(rolling_n).mean()
yCC = yCC.droplevel((0,1))

axs[0].plot(xCC.loc[cell_num,cycle]['Time'].to_numpy(), xCC.loc[cell_num,cycle]['Voltage'].to_numpy(), 'orange', label = 'after moving average')
axs[1].plot(xCC.loc[cell_num,cycle]['Voltage'].to_numpy(),  yCC.loc[cell_num,cycle].to_numpy(), 'orange', label = 'after moving average')


# temp = yCC.groupby(['cell_num', 'cycle'], sort = True).apply(lambda g: gaussian_filter1d(g, sigma=sigma))
# temp = (pd.concat(pd.DataFrame(temp.loc[(i)]) for i in temp.index).to_numpy().sum(axis = 1))
# yCC[yCC < 0.19] = temp[yCC < 0.19]
# 

axs[0].plot(xCC.loc[cell_num,cycle]['Time'].to_numpy(), xCC.loc[cell_num,cycle]['Voltage'].to_numpy(), 'g', label = 'after Gaussian filter')
axs[1].plot(xCC.loc[cell_num,cycle]['Voltage'].to_numpy(),  yCC.loc[cell_num,cycle].to_numpy(), 'g', label = 'after Gaussian filter')

plt.legend()


axs[1].set(xlabel = 'Cell voltage [V]', ylabel = 'Anode voltage vs. Li [V]')
axs[0].set(xlabel =  'Times [s]', ylabel = 'Cell voltage [V]' )

fig.suptitle('cell ' + str(cell_num) +  ', cycle ' + str(cycle) ) 

print(xCC.shape)
print(yCC.shape)

xCC.dropna(inplace = True, axis = 0)
inds = pd.isnull(yCC)
yCC.dropna(inplace = True, axis = 0)
xCC = xCC[~inds]

print(xCC.shape)
print(yCC.shape)

# print(xCC.tail())
del inds

using Li reference
(9798616, 6)
(9798616,)
(9713614, 6)
(9713614,)


In [13]:
print(masterInput.index.get_level_values(0).unique())
print(xCC.index.get_level_values(0).unique())
#The missing putput 10 here indicates the cells we should drop, go back to drop list, to reload the data, and drop list

Index([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
       37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
       55],
      dtype='int64', name='cell_num')
Index([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17, 18,
       19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36,
       37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54,
       55],
      dtype='int64', name='cell_num')


In [22]:
#create full features
from functions import FeatureEngineering
import importlib
importlib.reload(FeatureEngineering)

xCC, yCC = FeatureEngineering.create_charging_features(xCC, yCC, fRealReference, cell_params)
#if non-unique error occurs, 
    
print(xCC.shape)
print(yCC.shape)

(9719059, 22)
(9690659, 110)
(9690659,)


In [34]:
#create features without engineered features
from functions import FeatureEngineering_wo_engineered
import importlib
importlib.reload(FeatureEngineering_wo_engineered)

xCC, yCC = FeatureEngineering_wo_engineered.create_charging_features(xCC, yCC, fRealReference, cell_params)
#if non-unique error occurs, 
    
print(xCC.shape)
print(yCC.shape)

(20013095, 12)
(19984687, 60)
(19984687,)


In [17]:
#create features without lagged features
from functions import FeatureEngineering_wo_lagged
import importlib
importlib.reload(FeatureEngineering_wo_lagged)

xCC, yCC = FeatureEngineering_wo_lagged.create_charging_features(xCC, yCC, fRealReference, cell_params)
#if non-unique error occurs, 
    
print(xCC.shape)
print(yCC.shape)

ImportError: cannot import name 'FeatureEngineering_wo_lagged_simple' from 'functions' (unknown location)

In [18]:
#create features without lagged features and engineered features
from functions import FeatureEngineering_wo_lagged_engineered_simple
import importlib
importlib.reload(FeatureEngineering_wo_lagged_engineered_simple)

xCC, yCC = FeatureEngineering_wo_lagged_engineered_simple.create_charging_features(xCC, yCC, fRealReference, cell_params)
#if non-unique error occurs, 
    
print(xCC.shape)
print(yCC.shape)

(9719059, 11)
(9719059,)


In [24]:
# # Replace interp1d with UnivariateSpline
# import numpy as np
# # Your other code remains the same
# def calculate_dq_dv(cap, volt):
#     volt_np = np.array(volt)
#     cap_np = np.array(cap)
#     dq_dv = np.gradient(cap_np, volt_np)
#     return dq_dv
# funcs = xCC.groupby(xCC.index.names).apply(lambda g: calculate_dq_dv(g.Capacity, g.Voltage))
# print(funcs)

# # Your other code remains the same


c:\Users\Li group ML\anaconda3\envs\Feature_Engineering\lib\site-packages\numpy\lib\function_base.py:1239: RuntimeWarning: divide by zero encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
c:\Users\Li group ML\anaconda3\envs\Feature_Engineering\lib\site-packages\numpy\lib\function_base.py:1239: RuntimeWarning: invalid value encountered in divide
  a = -(dx2)/(dx1 * (dx1 + dx2))
c:\Users\Li group ML\anaconda3\envs\Feature_Engineering\lib\site-packages\numpy\lib\function_base.py:1240: RuntimeWarning: divide by zero encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
c:\Users\Li group ML\anaconda3\envs\Feature_Engineering\lib\site-packages\numpy\lib\function_base.py:1240: RuntimeWarning: invalid value encountered in divide
  b = (dx2 - dx1) / (dx1 * dx2)
c:\Users\Li group ML\anaconda3\envs\Feature_Engineering\lib\site-packages\numpy\lib\function_base.py:1241: RuntimeWarning: divide by zero encountered in divide
  c = dx1 / (dx2 * (dx1 + dx2))
c:\Users\Li group ML\anaconda3\envs\Feat

cell_num  cycle
1         2        [520.2913631651942, 284.831038230026, 0.0, 77....
          3        [0.0, 0.0, 403.9739062730223, 459.764401803135...
          4        [0.0, 286.6011746215663, 280.1512816898612, 0....
          5        [0.0, 387.0967741953791, 357.14285714352445, 0...
          6        [1152.073732722277, 812.5151588681401, 5.82076...
                                         ...                        
56        17       [24.9482324177334, 12.33264207118259, 13.77290...
          18       [25.320301817988874, 25.45148815572361, 12.667...
          19       [23.841880647546557, 25.10240682899098, 25.514...
          20       [24.493594924922746, 12.13891682606004, 12.464...
          21       [24.037305898751356, 24.33443382309838, 24.816...
Length: 3551, dtype: object


In [15]:
#create features without engineered features but with dq/dv
from functions import FeatureEngineering_wo_engineered_w_dvdq
import importlib
importlib.reload(FeatureEngineering_wo_engineered_w_dvdq)

xCC, yCC = FeatureEngineering_wo_engineered_w_dvdq.create_charging_features(xCC, yCC, fRealReference, cell_params)
#if non-unique error occurs, 
    
print(xCC.shape)
print(yCC.shape)

(9713614, 13)
(9685286, 65)
(9685286,)


In [16]:
print(date)

2024-04-15


In [17]:
#save data
if not os.path.exists(r'Features/' + date):
    os.makedirs(r'Features/' + date)

xCC.to_pickle(r'Features/' + date + r'/xCC.pkl')
yCC.to_pickle(r'Features/' + date + r'/yCC.pkl')

with open(r'Features/' + date + r'/masterList.pkl', 'wb') as f:
    pickle.dump(masterList, f)

In [18]:
from functions import TrainTestSplit
importlib.reload(TrainTestSplit)

#testing 3/15, LFP RE
# cell_test = [3, 5, 10]

# cell_test = [4,5]
# cell_test = [4,5,8,9]

#testing 12/1
# cell_test = [4,5]

#testing 2/8
# cell_test = [3,9,13]

#testing 2/16
# cell_test= [5,7]

#testing cherrypicking 
# cell_test = [2]

#testing 3/15, 3.55 activation
# cell_test = [1,2, 7]

#testing 3/15, Li
# cell_test = [3,5,6]

#testing 4/5, 3.55 activation
# cell_test = [3,5,7]

#testing 4/5, Canrd Li
# cell_test = [2,4,7]

#testing 4/19, 3.55 activation
# cell_test = [3,5, 9,12]

#testing 4/19, Canrd Li
# cell_test = [3,5,7]

#testing 4/19, all stepped in test
# cell_test = [11,12]

cell_test = [3,8,15,19,30,31,38,51]

cell_train = np.array([x for x in masterList.keys() if x not in cell_test])

xTrain, yTrain, xTest, yTest = TrainTestSplit.train_test_split(xCC, yCC, 'cell', cell_test, masterList)
timeTrain = xTrain['Time']
timeTest = xTest['Time']

#5


print(xTrain.shape, xTest.shape)
print(len(xTrain)/(len(xCC)))

Updated
Training list:  [1, 2, 4, 5, 6, 7, 9, 10, 11, 12, 13, 14, 16, 17, 18, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 52, 53, 54, 55]
Testing list:  [3, 8, 15, 19, 30, 31, 38, 51]
(7522996, 65) (2162290, 65)
0.7767448478031521


In [17]:
for cell_num in masterList.keys():
    idx = [xCC.index.get_level_values(0) == cell_num][0]
    print(cell_num, len(xCC[idx]))

1 816331
2 670954
3 814100
4 556116
5 590191
6 153451
7 442205
8 557814
9 286742
10 97075
11 285622
12 173530
13 179046
14 94132
15 145595
16 64588
17 11662
18 193011
19 76784
20 68790
21 16921
22 90403
23 121747
24 14255
25 6086
26 91061
27 203680
28 42481
29 314539
30 169390
31 18878
32 22120
33 233302
34 76891


## Data Visualization

In [ ]:
#plot cycle to cycle data for every cell
use_cols = ['Overpotential', 'Power', 'Current*Overpotential', 'Voltage-8', 'Current*Overpotential-2']
from functions import DataPlotting
importlib.reload(DataPlotting)

for i in masterList.keys():
    # i = 8
    try:
        DataPlotting.plot_multiple_cycles_one_cell(xCC, yCC, i, use_cols, masterList[i])
    except Exception as e:
        # print('failed ' + str(i))
        print(e)
    # break

c:\Users\Li group ML\Box\ArbinResultData\functions\DataPlotting.py:100: UserWarning: cannot plot cell #1, cycle 1
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))
c:\Users\Li group ML\Box\ArbinResultData\functions\DataPlotting.py:100: UserWarning: cannot plot cell #2, cycle 1
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))
c:\Users\Li group ML\Box\ArbinResultData\functions\DataPlotting.py:100: UserWarning: cannot plot cell #2, cycle 5
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))
c:\Users\Li group ML\Box\ArbinResultData\functions\DataPlotting.py:100: UserWarning: cannot plot cell #2, cycle 7
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))
c:\Users\Li group ML\Box\ArbinResultData\functions\DataPlotting.py:100: UserWarning: cannot plot cell #2, cycle 9
  warnings.warn("cannot plot cell #" + str(cellNum) + ", cycle " + str(cycle))
c:\Users\Li group ML\Box\ArbinResul

Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).
Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).
Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).
Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).
Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).
Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).
Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).
Locator attempting to generate 1810 ticks ([1, ..., 5428]), which exceeds Locator.MAXTICKS (1000).


In [31]:
#plot one cycle data for numerous cells
cols = ['Overpotential', 'Power', 'Current*Overpotential', 'Voltage-8', 'Current*Overpotential-2']
from functions import DataPlotting
importlib.reload(DataPlotting)
c_rate = 0.5
select_cells = masterList[masterList.values == str(c_rate)]
DataPlotting.plot_one_cycle_multiple_cells(xCC, yCC, 1, select_cells.keys(), [], cols)
plt.suptitle('Cycle 3, ' + str(c_rate)  + 'C')

failed 5 1
1
failed 6 1
1
failed 7 1
1
failed 8 1
1
failed 5 1
1
failed 6 1
1
failed 7 1
1
failed 8 1
1
failed 5 1
1
failed 6 1
1
failed 7 1
1
failed 8 1
1
failed 5 1
1
failed 6 1
1
failed 7 1
1
failed 8 1
1
failed 5 1
1
failed 6 1
1
failed 7 1
1
failed 8 1
1


Text(0.5, 0.98, 'Cycle 3, 0.5C')

In [36]:
#plot anode potential data
fig, axs = plt.subplots(10,5, figsize = (50,50))

w = 0.05
# volt_pop = xCC['Voltage']
volt_pop = yCC.copy()
n = math.ceil((volt_pop.max() -volt_pop.min())/w)
cell_num = 1
cycle = 3

for i in range(axs.shape[0]):
    for j in range(axs.shape[1]):
        # volt_sample = xCC.loc[cell_num, 5]['Voltage'].to_numpy()
        volt_sample = yCC.loc[cell_num,cycle].to_numpy()

        axs[i,j].hist(volt_sample, bins = n, label = cell_num)
        if(i == 1):
            axs[i,j].set_xlabel('Anode voltage')
        axs[i,j].set_title(str(cell_num) +  ', ' + str(masterList[cell_num]) + 'C')
        # axs[i,j].set_xticks([3.5, 3.75, 4, 4.25])
        axs[i,j].set_xticks([-0.1, 0, 0.1, 0.2, 0.3])
        cell_num+=1
        # break
        
# plt.ylabel('Frequency')
# plt.legend()
plt.figure()
plt.hist(volt_sample, bins = n)

#For RAM, AFTER YOU FINISHED the code here, click restart to decrease the RAM burden on it.

KeyError: (43, 3)